# Exploring EW4 data - GPM IMERG
This notebook explore the GPM IMERG dataset, including loading in PyEarthTools.

This dataset is a subset of the [NASA Integrated Multi-satellite Retrieval (IMERG)](https://gpm.nasa.gov/data/imerg) data for Global Precipitation Measurement (GPM). This subset has the following extents:
- Temporal Extent: April to September 2025
- Spatial Extent
  - Latitude 0N to 20N
  - Longitude 27W to 20E


### Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pathlib
import datetime
import functools
import math

In [ ]:
import numpy

In [ ]:
import xarray

In [ ]:
import matplotlib
import cartopy.crs

In [ ]:
import site_archive_jasmin

In [ ]:
import pyearthtools

In [ ]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe

In [ ]:
from pyearthtools.data import Petdt, TimeDelta
from pyearthtools.data.exceptions import DataNotFoundError
from pyearthtools.data.indexes import ArchiveIndex, decorators
from pyearthtools.data.transforms import Transform, TransformCollection
from pyearthtools.data.archive import register_archive


In [ ]:
from site_archive_jasmin.utilities import (
    cached_exists,
    cached_iterdir,
)  # Could these be moved into a generic module?


In [ ]:
import torch

## Explore the data in the dataset
Lets start by looking at a sample data file, before demonstrating loading the data in pyearthtools. From this we can learn the following things about the dataset
* The pattern of filenames for accessing the data
* The spatial and tempoeral extents of the data
* The variables present in the data
* Any potential problems with its usage, which can be fixed in the PyEarthTools data accessor class
* Plot some data so we can check that PyEarthTools loads it correctly.

We'll be using the IMERG data in the EW4 Group Workspace. It is also available through other sources:
* [NASA](https://gpm.nasa.gov/data/imerg)
* [AWS Open Data](https://registry.opendata.aws/nasa-gpm3imergm/)
* [CEDA Archive - available on JASMIN](https://catalogue.ceda.ac.uk/uuid/47c32530265d4d6e8fdb6c08b2330371/)


In [ ]:
ew4_gws_dir = pathlib.Path('/gws/nopw/j04/ew4energy/')
imerg_ew4_dir = ew4_gws_dir / 'imerg_2025_summer'


In [ ]:
def get_imerg_path(start_dt, time_delta, fname_template, data_dir):
    """
    Convenience function for constructing paths.
    """
    day_minutes = start_dt.hour * 60 + start_dt.minute
    date_str = '{dt.year:04d}{dt.month:02d}{dt.day:02d}'.format(dt=start_dt)
    time_template = '{dt.hour:02d}{dt.minute:02d}{dt.second:02d}'
    start_time = time_template.format(dt=start_dt)
    end_time = time_template.format(dt=start_dt+time_delta-datetime.timedelta(seconds=1))
    imerg_path = data_dir / fname_template.format(date_str=date_str,
                                              start_time=start_time,
                                              end_time=end_time,
                                                  day_minutes=day_minutes,
                                             )
    return imerg_path


In [ ]:
imerg_fname_template = '3B-HHR-E.MS.MRG.3IMERG.{date_str}-S{start_time}-E{end_time}.{day_minutes:04d}.V07B.HDF5.SUB.nc4'

In [ ]:
start_dt = datetime.datetime(2025,5,1,0,0)
end_dt = datetime.datetime(2025,6,1,0,0)
time_delta=datetime.timedelta(minutes=30)

In [ ]:
num_files = int((end_dt - start_dt) / time_delta)

In a month we have 2 files per hour, 24 hours per day for 31 days, 

In [ ]:
num_files

In [ ]:
imerg_filelist = [
    get_imerg_path(start_dt + (time_delta * time_ix),
                   time_delta,
                   imerg_fname_template,
                   imerg_ew4_dir
                  )
    for time_ix in range(num_files) ]


In [ ]:
ew4_imerge_sample_ds = xarray.open_mfdataset(imerg_filelist)

We have now loaded a month worth of precip data into a single xarray dataset.

In [ ]:
ew4_imerge_sample_ds['precipitation']


In [ ]:
float(min(ew4_imerge_sample_ds['lat'])), float(max(ew4_imerge_sample_ds['lat'])), float(min(ew4_imerge_sample_ds['lon'])), float(max(ew4_imerge_sample_ds['lon']))

In [ ]:
precip_plot_bins = [1e-3,0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]

To sanity check out loading of the data, we can plot a histogram to check we get a distribution that matches a typical distribution of precip data.

In [ ]:
ew4_imerge_sample_ds['precipitation'].plot.hist(bins=precip_plot_bins)


We can also plot the data using matplotlib to check we have data that matches our expectation

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(16,10))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
ew4_imerge_sample_ds['precipitation'][100].plot.contourf(ax=ax1)
ax1.coastlines(color='w')

## Loading the Data in PyEarthTools
Now we will load the data through the PyEarthTools data accessor. This is a class that has been created which how to access the data. For a dataset like this which is stored as files on disk, it specifries which files to load. Data could be loaded through other mechanisms though, for example it could be load from a database, through a web api, from a tape archive,  from a cloud-based object store or many other mechanisms.

We will take a look at the construction of the accessor, which can be found here:
* [IMERG Data Accessor](https://github.com/MetOffice/pyearthtools_jasmin/blob/main/src/site_archive_jasmin/ew4_imerg.py)
* [Template for Data Accessors](https://github.com/MetOffice/pyearthtools_jasmin/blob/main/src/site_archive_jasmin/template_accessor.py)

Further Reading 
* [
* [PyEarthTools and Data Access](https://pyearthtools.readthedocs.io/en/latest/data.html)
* [Data Module Deep Dive Tutorials](https://pyearthtools.readthedocs.io/en/latest/notebooks/Gallery.html#Deep-Dive---The-Data-Module)


In [ ]:
ew4_imerg_accessor = site_archive_jasmin.Ew4Imerg('2025-04-01 00:00', '2025-10-01 00:00')

In [ ]:
ew4_imerg_accessor

In [ ]:
ew4_imerg_accessor['2025-05-23 03:00']

In [ ]:
ew4_imerg_accessor['2025-05-23 13']



In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(16,10))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
ew4_imerg_accessor['2025-05-23 03:30']['precipitation'][0].plot.contourf(ax=ax1)

ax1.coastlines(color='w')


To demonstrate working with just a region of interest, we will select some data roughly corresponding to Ghana. The bounds have been tweaked a little so we can an array of a power of 2 size (64x64) as this is conveneient for working with a convolutional neural network.

In [ ]:
ghana_extents = {
    'latitude': (4.7,11.1),
    'longitude': (-3.5, 2.9),
}


In [ ]:
ghana_pet_box = (ghana_extents['latitude'][0],
                 ghana_extents['latitude'][1],
                 ghana_extents['longitude'][0],
                 ghana_extents['longitude'][1],
                )

In [ ]:
ew4_imerg_pipeline = petpipe.Pipeline(
    ew4_imerg_accessor,
    petdata.transform.region.Bounding(*ghana_pet_box),  
    petpipe.modifications.TemporalWindow(prior_indexes=[0,], posterior_indexes=[0,], timedelta=TimeDelta('30 minutes')),
    iterator=petpipe.iterators.DateRange('20250501T00', '20250511T00', interval='30 minutes'), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)


In [ ]:
ew4_imerg_pipeline['20250503T0700']

In [ ]:
ew4_imerg_iter = iter(ew4_imerg_pipeline)

In [ ]:
next(ew4_imerg_iter)

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(16,10))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
next(ew4_imerg_iter)[0][0]['precipitation'][0].plot.contourf(ax=ax1)
ax1.coastlines(color='w')


In [ ]:
next(ew4_imerg_iter), next(ew4_imerg_iter),

## Further Links

* [PyEarthTools Docs](https://pyearthtools.readthedocs.io/en/latest/)
  * [Tutorial Gallery](https://pyearthtools.readthedocs.io/en/latest/notebooks/Gallery.html)
* [PyEarthTools Repo](https://github.com/ACCESS-Community-Hub/PyEarthTools)
* [PyEarthTools JASMIN Site Archive Repo](https://github.com/MetOffice/pyearthtools_jasmin/)
*  [IMERG Dataset Info](https://gpm.nasa.gov/data/imerg)